Scenario: University Library Assistant

A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.

How It Works
- Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
- Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
- Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
- Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
- LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
- Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.

I will be using deepset/robert-base-squad2 model in this project


In [ ]:
!pip install chromadb

In [ ]:
!pip install sentence-transformers pypdf transformers

In [2]:
import os
from transformers import pipeline
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

Step 2: Load the document

In [3]:
reader=PdfReader('/content/Introduction_to_Data_Science.pdf')
text=' '
for page in reader.pages:
  text+=page.extract_text()

print('document Loaded')

document Loaded


Step 3: Chunk the document

In [4]:
print('Splitting document into chunks...')

def chunk_text(text,chunk_size=400,overlap=80):
  chunks=[]
  start=0

  while start<len(text):
    end=start+chunk_size
    chunk=text[start:end]
    chunks.append(chunk)
    start+=chunk_size-overlap

  return chunks
chunks=chunk_text(text)
print("Total chunks Created: ",len(chunks))

Splitting document into chunks...
Total chunks Created:  11


Step 4: Create embedding

In [5]:
embedding_model= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding Model Loaded')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


Step 5: Create vector database

In [6]:
client=chromadb.Client()
try:
  client.delete_collection('pdf_collection')
except:
  pass

collection=client.create_collection('pdf_collection')
print('New vector collection created')

New vector collection created


Step 6: Store chunks in database

In [7]:
for i, chunk in enumerate(chunks):
  embedding=embedding_model.encode(chunk).tolist()

  collection.add(documents=[chunk],
                 embeddings=[embedding],
                 ids=[str(i)])

print('All chunks are stored successfully')

All chunks are stored successfully


Step 7: Retriever Function (converts user questions -> embedding)

In [8]:
def retrieve(query,k=3):
  query_embedding=embedding_model.encode(query).tolist()

  results=collection.query(query_embeddings=[query_embedding],
                           n_results=k)

  return results['documents'][0]

Step 8: Load the LLM

In [9]:
qa_pipeline=pipeline("question-answering",
    model="deepset/roberta-base-squad2"
)

print('LLM loaded successfully')

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

LLM loaded successfully


Step 9: Question Answering Function

In [10]:
def answer_question(query):
  context_docs=retrieve(query)
  print("Retrieved docs:", context_docs)
  context=' '.join(context_docs)

  prompt=f'''You are an AI assistant.
  Answer the question using ONLY the context below. If the answer is not present,
  "Not found in document".

  Context:
  {context}

  Question:
  {query}

  Answer:
  '''
  response = qa_pipeline(
    question=query,
    context=context
)
  print("RAW MODEL OUTPUT:", response)
  return response['answer']

Step 10: Chat Loop

In [11]:
print("RAG Chatbot Ready")
print("Type 'exit' to stop")


while True:
  question = input('Ask a question: ')
  if question.lower()=='exit':
    print('Goodbye!')
    break

  answer=answer_question(question)

  print('\nAnswer:\n',answer)
  print('\n'+'-'*60+'\n')

RAG Chatbot Ready
Type 'exit' to stop
Ask a question: Explain the difference between supervised and unsupervised learning
Retrieved docs: ['ntation and clustering.\n\x7f\nReinforcement Learning – Algorithms learn through trial and error by receiving rewards or\npenalties.\nSupervised vs Unsupervised Learning\nSupervised learning requires labeled datasets, meaning the algorithm is trained with both inputs\nand expected outputs. It is commonly used for classification and regression tasks. In contrast,\nunsupervised learning works with unlabeled d', ' and regression tasks. In contrast,\nunsupervised learning works with unlabeled data. The goal is to discover patterns, clusters, or\nrelationships without predefined labels.\n4. Tools Used in Data Science\n\x7f\nPython – Most widely used programming language for data science.\n\x7f\nR – Popular for statistical computing.\n\x7f\nPandas – Data manipulation library.\n\x7f\nNumPy – Numerical computing library.\n\x7f\nMatplotlib', 'ning:\n\x7f\nS